# سبق 09 - میٹا کوگنیشن ڈیزائن پیٹرن


## سیٹ اپ

یہ نوٹ بک Microsoft Agent Framework استعمال کرتے ہوئے Metacognition ڈیزائن پیٹرن کی مثال پیش کرتی ہے۔

**ضروریات:**
- Azure OpenAI کی تعیناتی ماحولیاتی متغیرات کے ذریعے تشکیل شدہ ہو
- Azure CLI تصدیق شدہ ہو (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## میٹا ادراک کیا ہے؟

میٹا ادراک **سوچ کے بارے میں سوچنا** ہے۔

مصنوعی ذہانت کے ایجنٹس کے سیاق و سباق میں، اس کا مطلب ایسے ایجنٹس بنانا ہے جو کر سکیں:

- **خود پر غور کرنا** اپنے نتائج اور استدلالی عمل پر
- **غلطیوں کا پتا لگانا** اور خاموشی سے ناکام ہونے کے بجائے مناسب طریقے سے بحال ہونا
- **جانچنا** کہ آیا ان کے جوابات مکمل اور مددگار ہیں
- **اپنی حکمتِ عملی کو ڈھالنا** جب ابتدائی طریقہ کار کام نہ کرے (مثلاً، بیک اپ سسٹم پر واپس جانا)

ایک میٹا ادراکی ایجنٹ صرف سوالات کے جواب نہیں دیتا — یہ اپنی کارکردگی کی نگرانی کرتا ہے اور فوری طور پر حالات کے مطابق ڈھل جاتا ہے۔


## پرائمری اور بیک اپ ٹولز

ایک عام میٹا-ادراک کا پیٹرن **متبادل حکمتِ عملی** ہے۔ ایجنٹ پہلے پرائمری ٹول کو آزما تا ہے؛ اگر یہ ناکام ہو جائے (مثلاً، 404 خرابی)، ایجنٹ ناکامی کو پہچان کر شفاف طریقے سے بیک اپ ٹول پر سوئچ کر دیتا ہے۔

یہ حقیقی دنیا کے نظاموں کی عکاسی کرتا ہے جہاں پرائمری خدمات دستیاب نہیں ہو سکتیں اور ایجنٹس کو متبادل راستہ منتخب کرنے سے پہلے خود مسئلے کی تشخیص کرنی پڑتی ہے۔

ذیل میں ہم دو پرواز تلاش کرنے والے ٹولز کی تعریف کرتے ہیں:
- **پرائمری** — پیرس، ٹوکیو، اور بارسلونا کا احاطہ کرتا ہے
- **بیک اپ** — برلن، سڈنی، اور نیو یارک سٹی کا احاطہ کرتا ہے


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## خود عکاس ایجنٹ برائے خرابی کی بازیابی

نیچے دیے گئے ایجنٹ کو ہدایت کی گئی ہے کہ پہلے بنیادی پرواز کے نظام کو آزماۓ، ناکامیوں کو پہچانے، اور شفاف انداز میں بیک اپ نظام پر منتقل ہو جائے۔ ہر جواب کے بعد یہ مختصراً خود کا جائزہ لیتا ہے کہ آیا اس نے صارف کے سوال کا مکمل جواب دیا ہے یا نہیں۔


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## خود تشخیصی نمونہ

مابعدِ ادراک کا ایک اور پہلو **خود تشخیصی** ہے: ایک علیحدہ ایجنٹ (یا اسی ایجنٹ کے ذریعے دوسری بار) ایک جواب کا جائزہ لیتا ہے کہ آیا وہ مکمل، درست، اور مددگار ہے۔

ذیل میں ہم ایک `ResponseEvaluator` ایجنٹ بناتے ہیں جو سفر ایجنٹ کے جوابات کو تین جہتوں پر اسکور کرتا ہے۔


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## خلاصہ

اس سبق میں آپ نے سیکھا کہ Microsoft Agent Framework استعمال کرتے ہوئے **مابعد ادراکی ایجنٹس** کیسے بنائے جائیں:

- **خود عکاسی**: ایسے ایجنٹس جو اپنے استدلال کی نگرانی کرتے ہیں اور شفاف انداز میں بتاتے ہیں کہ کیا ہوا۔
- **فیل بیک کے ساتھ خرابی کی بازیابی**: ایک بنیادی + بیک اپ ٹول پیٹرن جس میں ایجنٹ ناکامیاں (مثلاً 404 errors) معلوم کرتا ہے اور خود بخود کسی متبادل ماخذ کو آزمانے کی کوشش کرتا ہے۔
- **خود تشخیص**: ایک الگ جانچنے والا ایجنٹ جو جوابات کو تکمیل، درستگی، اور مفیدیت کے لحاظ سے اسکور کرتا ہے۔

یہ پیٹرن ایجنٹس کو زیادہ مضبوط، شفاف، اور قابلِ اعتماد بناتے ہیں — جو پروڈکشن تعیناتیوں کے لیے اہم خصوصیات ہیں۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
اعلانِ عدمِ ذمہ داری:

اس دستاویز کا ترجمہ مصنوعی ذہانت کی ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے کیا گیا ہے۔ اگرچہ ہم درستگی کے لیے کوشاں ہیں، براہِ کرم نوٹ کریں کہ خودکار تراجم میں غلطیاں یا عدمِ درستی ہو سکتی ہے۔ اصل دستاویز کو اس کی مادری زبان میں معتبر ماخذ سمجھا جانا چاہیے۔ اہم معلومات کے لیے پیشہ ورانہ انسانی ترجمہ تجویز کیا جاتا ہے۔ ہم اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تعبیر کے لیے ذمہ دار نہیں ہیں۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
